In [ ]:
#importer les bibliothèques nécessaires
from vectorbtpro import * 
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import optuna

In [ ]:
#Charger les données à partir d'un fichier CSV
data = pd.read_csv('../data/raw/BTCUSDT.csv')

#Conversion timestamp ms 
data['date'] = pd.to_datetime(data['date'], unit='ms')
data = data.set_index('date')

##

# start = pd.Timestamp('2025-12-01')
# end = pd.Timestamp('2026-01-25')

# data = data.loc[start:end]

In [ ]:
#Paramètres du Keltner Channel
ma_window = 200 #Période de l'EMA
atr_window = 14 #Période de l'ATR
atr_mult = 2 #Multiplicateur ATR pour les bandes

#Calcul de l'EMA avec VectorBT PRO (Keltner utilise une EMA, pas une SMA)
ema = vbt.MA.run(data['close'], window=ma_window, wtype='Exp').ma
atr = vbt.ATR.run(data['high'], data['low'], data['close'], window=atr_window).atr

#Shift de l'EMA et des bandes pour éviter le look-ahead bias
ema_shifted = ema.shift(1)
atr_shifted = atr.shift(1)
bande_inf_shifted = ema_shifted - atr_mult * atr_shifted
bande_sup_shifted = ema_shifted + atr_mult * atr_shifted

In [ ]:
#--- Signaux Long ---
long_entries = data['low'] <= bande_inf_shifted
long_exits = data['high'] >= ema_shifted

#--- Signaux Short ---
short_entries = data['high'] >= bande_sup_shifted
short_exits = data['low'] <= ema_shifted

#--- Prix d'exécution custom ---
price = pd.Series(np.nan, index=data.index)
price[long_exits] = ema_shifted[long_exits]
price[short_exits] = ema_shifted[short_exits]
price[long_entries] = bande_inf_shifted[long_entries]
price[short_entries] = bande_sup_shifted[short_entries]

In [ ]:
pf = vbt.Portfolio.from_signals(
    data['close'],
    long_entries=long_entries,
    long_exits=long_exits,
    short_entries=short_entries,
    short_exits=short_exits,
    price=price,
    open=data['open'],
    high=data['high'],
    low=data['low'],
    size=1,
    size_type='valuepercent',
    fees=0.001,
    init_cash=10000,
    accumulate=False,
)
pf.stats()

In [ ]:
# #Visualisation avec VectorBT PRO
# fig = pf.plot()
# fig.add_trace(go.Scatter(x=data.index, y=ema_shifted, name='EMA', line=dict(color='orange')))
# fig.add_trace(go.Scatter(x=data.index, y=bande_sup_shifted, name='KC Sup', line=dict(color='green', dash='dash'), opacity=0.5))
# fig.add_trace(go.Scatter(x=data.index, y=bande_inf_shifted, name='KC Inf', line=dict(color='red', dash='dash'), opacity=0.5))
# fig.show()

In [ ]:
#On a fait un bt simple, avec des paramètres lambda pour pouvoir les opti avec Optuna par la suite. Et faire un walkforward pour tester la robustesse du système.
#Le backtest n'est pas concluant.
###Mise en place du walkforward et de l'optimisation avec Optuna

#Définir la stratégie sous forme d'une fonction

def run_backtest(data, ma_window, atr_window, atr_mult, sl_stop=None, size=0.3):
    #Keltner Channel = EMA ± multiplicateur × ATR
    ema = vbt.MA.run(data['close'], window=ma_window, wtype='Exp').ma
    atr = vbt.ATR.run(data['high'], data['low'], data['close'], window=atr_window).atr
    
    ema_shifted = ema.shift(1)
    atr_shifted = atr.shift(1)
    bande_inf = ema_shifted - atr_mult * atr_shifted
    bande_sup = ema_shifted + atr_mult * atr_shifted

    long_entries = data['low'] <= bande_inf
    long_exits = data['high'] >= ema_shifted
    short_entries = data['high'] >= bande_sup
    short_exits = data['low'] <= ema_shifted

    price = pd.Series(np.nan, index=data.index)
    price[long_exits] = ema_shifted[long_exits]
    price[short_exits] = ema_shifted[short_exits]
    price[long_entries] = bande_inf[long_entries]
    price[short_entries] = bande_sup[short_entries]

    pf_kwargs = dict(
        close=data['close'],
        long_entries=long_entries, long_exits=long_exits,
        short_entries=short_entries, short_exits=short_exits,
        price=price,
        open=data['open'], high=data['high'], low=data['low'],
        size=size, size_type='valuepercent',
        fees=0.001, init_cash=10000, accumulate=False,
    )
    
    #sl_stop = pourcentage de perte max avant fermeture auto du trade
    #ex: sl_stop=0.05 → si le trade perd 5%, on ferme
    if sl_stop is not None:
        pf_kwargs['sl_stop'] = sl_stop
    
    pf = vbt.Portfolio.from_signals(**pf_kwargs)
    return pf

In [ ]:
#Utiliser tout le dataset pour l'optimisation
data_full = pd.read_csv('../data/raw/BTCUSDT.csv')
data_full['date'] = pd.to_datetime(data_full['date'], unit='ms')
data_full = data_full.set_index('date')

In [ ]:
def get_wf_windows(data, train_days=180, test_days=60, step_days=60):
    """
    Découpe les données en fenêtres walk-forward.
    Step = test_days pour éviter le chevauchement des tests.
    Retourne une liste de tuples (train_data, test_data).
    """
    windows = []
    start = data.index[0]
    end = data.index[-1]

    #Convertir les jours en Timedelta
    train_delta = pd.Timedelta(days=train_days)
    test_delta = pd.Timedelta(days=test_days)
    step_delta = pd.Timedelta(days=step_days)  #step = test pour que l'intersection est nulle
    
    current = start #current comme curseur pour découper les datasets
    while current + train_delta + test_delta <= end:
        train_start = current
        train_end = current + train_delta

        test_start = train_end
        test_end = train_end + test_delta
        
        train_data = data.loc[train_start:train_end]
        test_data = data.loc[test_start:test_end]
        
        windows.append((train_data, test_data))
        current += step_delta
    
    return windows #retourne la liste des tuples (train, test)

#Calculer le nombre de jours pour avoir 10 fenêtres sur les données
data_length_days = (data_full.index[-1] - data_full.index[0]).days
print(f"Longueur totale des données: {data_length_days} jours")

# Calculer le nombre de jours pour avoir 10 fenêtres
window_length_days = data_length_days // 10
print(f"Longueur de chaque fenêtre: {window_length_days} jours")

#On veut par fenêtres 2/3 de train et 1/3 de test, soit 20 jours de train et 10 jours de test pour une fenêtre de 30 jours
train_days = window_length_days * 2 // 3
test_days = window_length_days - train_days
step_days = (data_length_days - train_days) // 10 
print(f"Train: {train_days} jours, Test: {test_days} jours, avance de {step_days} jours pour chaque fenêtre")

windows = get_wf_windows(data_full, train_days, test_days, step_days=step_days) #step = test pour que les fenêtres de test ne se chevauchent pas
print(f"Nombre de fenêtres: {len(windows)}")
for i, (train, test) in enumerate(windows): #enumerate pour afficher le numéro de la fenêtre
    print(f"Fenêtre {i+1}: Train [{train.index[0].date()} → {train.index[-1].date()}] Test [{test.index[0].date()} → {test.index[-1].date()}]")

In [ ]:
n_windows = len(windows)
results = [] #meilleur resultat de chaque fenêtres 
all_trials_data = []  #stocke tout les trials pour l'analyse

for i, (train_data, test_data) in enumerate(windows): #on boucle sur chaque fenêtre que get_wf_windows a créé avec i comme compteur de fenêtres 
    print(f"\n--- Fenêtre {i+1}/{n_windows} ---")
    print(f"Train: {train_data.index[0].date()} → {train_data.index[-1].date()}")
    print(f"Test:  {test_data.index[0].date()} → {test_data.index[-1].date()}")
    
    def objective(trial):
        pf = run_backtest(train_data,
                          ma_window=trial.suggest_int('ma_window', 10, 500, step=5),
                          atr_window=trial.suggest_int('atr_window', 5, 100, step=5),
                          atr_mult=trial.suggest_float('atr_mult', 0.5, 10.0, step=0.25),
                          sl_stop=trial.suggest_float('sl_stop', 0.01, 0.15, step=0.01)) #sl_stop = stop loss entre 1% et 15%
        
        total_trades = pf.trades.count()
        if total_trades < 30: #moins de 30 trades dans une fenêtre = poubelle
            return -999
        if pf.max_drawdown > 0.30: #drawdown max 30% trop risqué
            return -999
        
        return pf.sharpe_ratio * 0.7 + pf.total_return * 0.3 * (1 - pf.max_drawdown) #definition du score avec des poids pour privilégier le sharpe et pénaliser les gros drawdowns
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=200, show_progress_bar=True) #optimisation sur la fenêtre courante
    
    best_params = study.best_params #récupérer les meilleurs paramètres de la fenêtre courante
    print(f"Meilleurs params: ma_window={best_params['ma_window']}, atr_window={best_params['atr_window']}, atr_mult={best_params['atr_mult']:.2f}, sl_stop={best_params['sl_stop']:.2f}")
    
    #Stocker tout les trials de cette fenêtre
    for t in study.trials: #pour chaque trial de la fenêtre courante
        all_trials_data.append({ #stocker les paramètres 
            'window': i + 1,
            'ma_window': t.params['ma_window'],
            'atr_window': t.params['atr_window'],
            'atr_mult': t.params['atr_mult'],
            'sl_stop': t.params['sl_stop'],
            'score': t.value,
        })
    
    pf_test = run_backtest(test_data, best_params['ma_window'], best_params['atr_window'], best_params['atr_mult'], best_params['sl_stop']) #on bt avec le meilleurs paramètres trouvées par train sur la partie test de la fenêtre courante
    
    results.append({ #stocker les résultats de la fenêtre courante
        'window': i + 1,
        'ma_window': best_params['ma_window'],
        'atr_window': best_params['atr_window'],
        'atr_mult': best_params['atr_mult'],
        'sl_stop': best_params['sl_stop'],
        'train_sharpe': study.best_value,
        'test_sharpe': pf_test.sharpe_ratio,
        'test_return': pf_test.total_return,
        'test_max_dd': pf_test.max_drawdown,
        'test_trades': pf_test.trades.count(),
    })

In [ ]:
#Mise en forme des résultats pour l'analyse
df_results = pd.DataFrame(results) #tableau des meilleurs essais par fenêtre
df_trials = pd.DataFrame(all_trials_data) #tableau de tout les essais de toutes les fenêtres
n_windows = len(windows) #nombre total de fenêtres

#on garde que les essais valides 
valid = df_trials[df_trials['score'] > -999].copy()

#on récupère le min et max de chaque paramètre pour normaliser
ma_min, ma_max = valid['ma_window'].min(), valid['ma_window'].max()
atrw_min, atrw_max = valid['atr_window'].min(), valid['atr_window'].max()
atrm_min, atrm_max = valid['atr_mult'].min(), valid['atr_mult'].max()
sl_min, sl_max = valid['sl_stop'].min(), valid['sl_stop'].max()

#normalisation entre 0 et 1 pour que la distance soit comparable entre les paramètres
#car ma_window (10-500) écraserait sl_stop (0.01-0.15) dans le calcul de distance ((valeur - min) / (max - min))
valid['ma_norm'] = (valid['ma_window'] - ma_min) / (ma_max - ma_min) if ma_max > ma_min else 0 #éviter division par zéro si tous les essais ont la même valeur pour ce paramètre
valid['atrw_norm'] = (valid['atr_window'] - atrw_min) / (atrw_max - atrw_min) if atrw_max > atrw_min else 0
valid['atrm_norm'] = (valid['atr_mult'] - atrm_min) / (atrm_max - atrm_min) if atrm_max > atrm_min else 0
valid['sl_norm'] = (valid['sl_stop'] - sl_min) / (sl_max - sl_min) if sl_max > sl_min else 0

#pour chaque fenêtre, garder le top 30% des essais
top_per_window = {} #clé = numéro de fenêtre, valeur = tableau des meilleurs essais.
for w in range(1, n_windows + 1):
    wt = valid[valid['window'] == w] #tous les essais valides de la fenêtre w
    threshold = wt['score'].quantile(0.70) #seuil = score au dessus duquel on est dans le top 30%
    top_per_window[w] = wt[wt['score'] >= threshold].copy() #on garde que les essais du top 30% de la fenêtre w pour l'analyse de stabilité

#4 dimensions → seuil plus large que en 3D (distance max en 4D = √4 = 2.0)
seuil = 0.35
points_to_plot = []


for w, top in top_per_window.items(): #pour chaque fenêtres
    for _, row in top.iterrows(): #pour chaque essai du top 30% de cette fenêtre
        nearby_windows = 0
        
        #on regarde les AUTRES fenêtres
        for w2, top2 in top_per_window.items():
            if w2 == w:
                continue #on skip sa propre fenêtre
            
            #distance euclidienne 4D entre cet essai et tous les bons essais de l'autre fenêtre
            distances = np.sqrt(
                (top2['ma_norm'] - row['ma_norm'])**2 + 
                (top2['atrw_norm'] - row['atrw_norm'])**2 + 
                (top2['atrm_norm'] - row['atrm_norm'])**2 +
                (top2['sl_norm'] - row['sl_norm'])**2
            )
            
            #si au moins 1 bon essai de l'autre fenêtre est à moins de 35% on à trouvé un voisin 
            if (distances <= seuil).any():
                nearby_windows += 1
        ###
        total_other = n_windows - 1 #nombre de fenêtres autres que la sienne
        ratio = nearby_windows / total_other #proportion de fenêtres qui ont un voisin proche
        
        #on garde que si au moins 50% des autres fenêtres ont un voisin
        if ratio >= 0.50:
            points_to_plot.append({
                'ma_window': row['ma_window'],
                'atr_window': row['atr_window'],
                'atr_mult': row['atr_mult'],
                'sl_stop': row['sl_stop'],
                'score': row['score'],
                'window': w,
                'nearby_windows': nearby_windows,
                'ratio': ratio,
                'color': 'green' if ratio >= 0.70 else 'orange' #vert si +70% et orange si 50-70%
            })

df_points = pd.DataFrame(points_to_plot) #tableau des points stables à afficher

In [ ]:
fig = go.Figure()

#Affichage des points stables
if len(df_points) == 0:
    #si aucun point a survécu au filtre précedent les params sont trop différents entre fenêtres
    print("Aucun point stable trouvé — les paramètres sont trop instables entre fenêtres")
else:
    #on trace les points orange et vert séparément pour la légende
    for color, label in [('orange', '50-70% fenêtres'), ('green', '> 70% fenêtres')]: #on boucle sur les 2 couleurs pour faire 2 traces différentes (orange et vert)
        subset = df_points[df_points['color'] == color] #on prend que les points de la couleur courante
        if len(subset) == 0: 
            continue #i y'a pas de points de cette couleur on skip
        fig.add_trace(go.Scatter(
            x=subset['ma_window'], y=subset['atr_mult'], mode='markers',
            marker=dict(color=color, size=10, opacity=0.6),
            name=label,
            #texte au survol de la souris
            text=[f"Score: {s:.2f} | atr_window: {aw} | sl: {sl:.2f} | Voisins: {n}/{n_windows-1}" 
                  for s, aw, sl, n in zip(subset['score'], subset['atr_window'], subset['sl_stop'], subset['nearby_windows'])]
        )) 
    
    fig.update_layout(
        title=f'Zones robustes cross-fenêtres (seuil {seuil:.0%})',
        xaxis_title='ma_window', yaxis_title='atr_mult',
    )
    fig.show()
    ###
    
    #on prend le vert si y'en a, sinon l'orange
    best_color = 'green' if len(df_points[df_points['color'] == 'green']) > 0 else 'orange'
    cluster = df_points[df_points['color'] == best_color]
    
    #params finaux = médiane du meilleur cluster
    final_ma = int(cluster['ma_window'].median())
    final_atrw = int(cluster['atr_window'].median())
    final_atrm = cluster['atr_mult'].median()
    final_sl = cluster['sl_stop'].median()
    print(f"\nParamètres finaux (centre du cluster {best_color}): ma_window={final_ma}, atr_window={final_atrw}, atr_mult={final_atrm:.2f}, sl_stop={final_sl:.2f}")

In [ ]:
# #Affichage des plages de paramètres des meilleurs essais stables pour chaque fenêtre
# for w in range(1, n_windows + 1):
#     wt = valid[valid['window'] == w]
#     threshold = wt['score'].quantile(0.70)
#     top = wt[wt['score'] >= threshold]
#     print(f"Fenêtre {w}: ma [{top['ma_window'].min()}-{top['ma_window'].max()}] atr_w [{top['atr_window'].min()}-{top['atr_window'].max()}] atr_m [{top['atr_mult'].min()}-{top['atr_mult'].max()}] sl [{top['sl_stop'].min()}-{top['sl_stop'].max()}]")

In [ ]:
#backtest sur TOUT le dataset avec les params trouvés par l'analyse de clusters
pf_final = run_backtest(data_full, ma_window=final_ma, atr_window=final_atrw, atr_mult=final_atrm, sl_stop=final_sl, size=1)
print(pf_final.stats())

ema_final = vbt.MA.run(data_full['close'], window=final_ma, wtype='Exp').ma
ema_final_shifted = ema_final.shift(1)
atr_final = vbt.ATR.run(data_full['high'], data_full['low'], data_full['close'], window=final_atrw).atr
atr_final_shifted = atr_final.shift(1)
bande_inf_final = ema_final_shifted - final_atrm * atr_final_shifted
bande_sup_final = ema_final_shifted + final_atrm * atr_final_shifted

fig = pf_final.plot()
fig.add_trace(go.Scatter(x=data_full.index, y=ema_final_shifted, name='EMA', line=dict(color='orange')))
fig.add_trace(go.Scatter(x=data_full.index, y=bande_sup_final, name='KC Sup', line=dict(color='green', dash='dash'), opacity=0.5))
fig.add_trace(go.Scatter(x=data_full.index, y=bande_inf_final, name='KC Inf', line=dict(color='red', dash='dash'), opacity=0.5))
fig.show()

In [ ]:
### Scatter 3D des paramètres (clusters)
#visualisation en 3D de l'espace des paramètres du top 30% de chaque fenêtre
#chaque couleur = une fenêtre, les points proches entre fenêtres = zone robuste

fig = go.Figure() #graphique plotly vide
colors = px.colors.qualitative.Set3 #palette de 12 couleurs prédéfinies, une par fenêtre

for w in range(1, n_windows + 1): #on boucle sur chaque fenêtre (1 à 10)
    top = top_per_window[w] #top 30% des essais de la fenêtre w
    fig.add_trace(go.Scatter3d( #ajouter un nuage de points 3D pour cette fenêtre
        x=top['ma_window'], y=top['atr_window'], z=top['atr_mult'], #un axe par paramètre
        mode='markers', #que des points, pas de lignes
        marker=dict(size=5, color=colors[(w-1) % len(colors)], opacity=0.6), #taille 5, couleur unique par fenêtre, un peu transparent
        name=f'F{w}', #nom dans la légende (F1, F2...)
        text=[f"Score: {s:.2f} | sl: {sl:.2f}" for s, sl in zip(top['score'], top['sl_stop'])] #texte au survol avec le sl
    ))

#ajouter les params finaux en gros point noir
fig.add_trace(go.Scatter3d(
    x=[final_ma], y=[final_atrw], z=[final_atrm], #un seul point = la médiane du cluster
    mode='markers',
    marker=dict(size=10, color='black', symbol='diamond'), #gros diamant noir pour qu'il ressorte
    name=f'Param final (sl={final_sl:.2f})' #nom dans la légende avec le sl
))

fig.update_layout(
    title='Espace des paramètres 3D — Top 30% par fenêtre',
    scene=dict(xaxis_title='ma_window', yaxis_title='atr_window', zaxis_title='atr_mult'), #labels des 3 axes (scene = config 3D)
    height=700 #hauteur du graphique en pixels
)
fig.show() #afficher, on peut tourner le graphique avec la souris